# 04f: Deployment-safe stacked model (XGBoost + FT-Transformer `nn_score`)

Notebook 04e proved the `nn_score` stacking gain is real (isolated +0.0853 PR-AUC on top of
the extra-data effect), but it trained on `train_holdout + val` combined -- val is the
notebook 05 calibration set, so that model can't be threshold-calibrated honestly. This
notebook trains the *deployment* stacked model on `train_holdout` only, keeping `val`
completely clean, then re-runs notebook 05's full calibration procedure (cost sweep,
98%-specificity target, category-aware James-Stein shrinkage) against this model's own
score distribution before touching `test`.

This produces the actual artifacts served by `src/api/`: `models/fraud_xgboost.json`
(22 features, `nn_score` included), `models/feature_columns.json`, `models/threshold.json`.
`data/processed/feature_columns.json` (the 21-feature research config notebooks 03/04b-e
read) is untouched -- these are deliberately different files for different purposes.

In [1]:
import os, json
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, confusion_matrix, classification_report,
)
import mlflow
import mlflow.xgboost
from mlflow.tracking import MlflowClient

with open("../data/processed/feature_columns.json") as f:
    cfg = json.load(f)
FEATURES, LABEL = cfg["features"], cfg["label"]
AUG_FEATURES = FEATURES + ["nn_score"]

train_full = pd.read_parquet("../data/processed/train.parquet")
val = pd.read_parquet("../data/processed/val.parquet").reset_index(drop=True)
test = pd.read_parquet("../data/processed/test.parquet").reset_index(drop=True)
nn_scores = pd.read_parquet("../reports/nn_scores.parquet")

def attach_nn_score(df, split_name):
    ids = nn_scores.loc[nn_scores.split == split_name, ["transaction_id", "nn_score"]]
    out = df.merge(ids, on="transaction_id", how="inner")
    assert len(out) == len(df), f"{split_name}: lost rows joining nn_score ({len(out)} vs {len(df)})"
    return out

train_holdout = attach_nn_score(
    train_full[train_full.transaction_id.isin(
        nn_scores.loc[nn_scores.split == "train_holdout", "transaction_id"])],
    "train_holdout",
).reset_index(drop=True)
val_aug = attach_nn_score(val, "val")
test_aug = attach_nn_score(test, "test")

print(f"train_holdout={train_holdout.shape}  val={val_aug.shape}  test={test_aug.shape}")
print(f"train_holdout fraud rate={train_holdout[LABEL].mean():.4%}  "
      f"val={val_aug[LABEL].mean():.4%}  test={test_aug[LABEL].mean():.4%}")

train_holdout=(614002, 28)  val=(157287, 28)  test=(157286, 28)
train_holdout fraud rate=0.5972%  val=0.4476%  test=0.6059%


In [2]:
os.makedirs("../models", exist_ok=True)
mlflow.set_tracking_uri(f"sqlite:///{os.path.abspath('../mlflow.db')}")
mlflow.set_experiment("fraud-detection")

X_train, y_train = train_holdout[AUG_FEATURES], train_holdout[LABEL]
X_val, y_val = val_aug[AUG_FEATURES], val_aug[LABEL]
X_test, y_test = test_aug[AUG_FEATURES], test_aug[LABEL]

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Same hyperparameters as the currently-deployed (non-stacked) model -- isolates the effect
# of adding nn_score + train-only rows rather than confounding it with a fresh sweep.
STACKED_XGB_PARAMS = dict(
    max_depth=6, learning_rate=0.05, n_estimators=500,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5, reg_lambda=5.0,
)

with mlflow.start_run(run_name="xgb_stacked_deploy_train_holdout_only") as run:
    stacked_model = xgb.XGBClassifier(
        **STACKED_XGB_PARAMS,
        scale_pos_weight=scale_pos_weight, objective="binary:logistic",
        eval_metric="aucpr", random_state=42, n_jobs=-1, tree_method="hist",
    )
    stacked_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    val_proba = stacked_model.predict_proba(X_val)[:, 1]
    test_proba = stacked_model.predict_proba(X_test)[:, 1]
    roc_auc = roc_auc_score(y_test, test_proba)
    pr_auc = average_precision_score(y_test, test_proba)
    val_roc = roc_auc_score(y_val, val_proba)
    val_pr = average_precision_score(y_val, val_proba)

    mlflow.log_params({**STACKED_XGB_PARAMS, "scale_pos_weight": round(scale_pos_weight, 2),
                        "model": "xgboost_stacked", "features": "21_base+nn_score",
                        "train_rows": len(X_train)})
    mlflow.log_metrics({"val_roc_auc": val_roc, "val_pr_auc": val_pr,
                         "test_roc_auc": roc_auc, "test_pr_auc": pr_auc})
    mlflow.xgboost.log_model(stacked_model, "model")
    best_run_id = run.info.run_id

print(f"Deployment-safe stacked model (train_holdout only, {len(X_train):,} rows):")
print(f"  val  ROC-AUC={val_roc:.4f}  val  PR-AUC={val_pr:.4f}")
print(f"  test ROC-AUC={roc_auc:.4f}  test PR-AUC={pr_auc:.4f}")
print(f"  (research number from 04e, trained on train_holdout+val: test PR-AUC=0.8758 -- "
      f"expect this to be somewhat lower since it saw {len(val_aug):,} fewer rows)")

registered = mlflow.register_model(f"runs:/{best_run_id}/model", "fraud-xgboost")
client = MlflowClient()
client.set_registered_model_alias("fraud-xgboost", "staging", registered.version)
print(f"Registered 'fraud-xgboost' version {registered.version} -> alias 'staging'")

2026/08/31 11:27:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Registered model 'fraud-xgboost' already exists. Creating a new version of this model...
2026/08/31 11:27:08 WARNING mlflow.tracking._model_registry.fluent: Run with id 42dfffa5b3674f93911947ee6974b9ff has no artifacts at artifact path 'model', registering model based on models:/m-c53ac75c793042b9b3616a25db87f662 instead


Deployment-safe stacked model (train_holdout only, 614,002 rows):
  val  ROC-AUC=0.9959  val  PR-AUC=0.8425
  test ROC-AUC=0.9965  test PR-AUC=0.8716
  (research number from 04e, trained on train_holdout+val: test PR-AUC=0.8758 -- expect this to be somewhat lower since it saw 157,287 fewer rows)
Registered 'fraud-xgboost' version 16 -> alias 'staging'


Created version '16' of model 'fraud-xgboost'.


In [3]:
REVIEW_COST = 5.0
val_amounts = val_aug["amount"].values
y_val_arr = val_aug[LABEL].values
amounts = test_aug["amount"].values
y_arr = y_test.values

no_model_cost = amounts[y_arr == 1].sum()
review_all_cost = REVIEW_COST * len(test_aug)

thresholds = np.linspace(0.01, 0.99, 99)
total_costs, fn_costs, fp_costs = [], [], []
for t in thresholds:
    pred = (val_proba >= t).astype(int)
    fn_mask = (pred == 0) & (y_val_arr == 1)
    fp_mask = (pred == 1) & (y_val_arr == 0)
    fn_cost = val_amounts[fn_mask].sum()
    fp_cost = REVIEW_COST * fp_mask.sum()
    fn_costs.append(fn_cost); fp_costs.append(fp_cost); total_costs.append(fn_cost + fp_cost)

total_costs = np.array(total_costs)
cost_min_idx = total_costs.argmin()
cost_min_threshold = thresholds[cost_min_idx]

fpr, tpr, roc_thresholds = roc_curve(y_val_arr, val_proba)
TARGET_SPECIFICITY = 0.98
max_fpr = 1 - TARGET_SPECIFICITY
valid = fpr <= max_fpr
spec_idx = np.where(valid)[0][-1]
best_threshold = float(roc_thresholds[spec_idx])
achieved_specificity_val = 1 - fpr[spec_idx]
achieved_recall_val = tpr[spec_idx]

pred_at_best = (test_proba >= best_threshold).astype(int)
fn_mask_best = (pred_at_best == 0) & (y_arr == 1)
fp_mask_best = (pred_at_best == 1) & (y_arr == 0)
cost_at_best = amounts[fn_mask_best].sum() + REVIEW_COST * fp_mask_best.sum()

print(f"Specificity-targeted threshold (calibrated on clean val): {best_threshold:.4f}")
print(f"  achieved specificity (val): {achieved_specificity_val:.4f}  (target {TARGET_SPECIFICITY})")
print(f"  achieved recall (val):      {achieved_recall_val:.4f}")
print(f"Expected cost @ this threshold, on TEST: ${cost_at_best:,.0f}")
print(f"Cost of approving everything (no model), on TEST: ${no_model_cost:,.0f}")
print(f"Estimated savings vs. no model (on TEST): ${no_model_cost - cost_at_best:,.0f} "
      f"({(1 - cost_at_best / no_model_cost):.1%} reduction)")

Specificity-targeted threshold (calibrated on clean val): 0.0422
  achieved specificity (val): 0.9810  (target 0.98)
  achieved recall (val):      0.9602
Expected cost @ this threshold, on TEST: $17,040
Cost of approving everything (no model), on TEST: $511,283
Estimated savings vs. no model (on TEST): $494,243 (96.7% reduction)


In [4]:
MIN_LEGIT_FOR_CATEGORY_CALIBRATION = 1000

val_eval = val_aug.copy()
val_eval["proba"] = val_proba

category_thresholds = {}
calib_rows = []
for cat, grp in val_eval.groupby("category"):
    legit_scores = grp.loc[grp.is_fraud == 0, "proba"].values
    fraud_scores = grp.loc[grp.is_fraud == 1, "proba"].values

    if len(legit_scores) >= MIN_LEGIT_FOR_CATEGORY_CALIBRATION:
        cat_threshold = float(np.quantile(legit_scores, TARGET_SPECIFICITY))
    else:
        cat_threshold = best_threshold
    category_thresholds[cat] = cat_threshold

    calib_rows.append({
        "category": cat, "n_legit_val": len(legit_scores), "n_fraud_val": len(fraud_scores),
        "raw_threshold": cat_threshold,
    })

category_calib = pd.DataFrame(calib_rows).set_index("category")

# Same James-Stein shrinkage as notebook 05: categories with few fraud examples in val pull
# toward the global threshold rather than overfitting to a handful of points.
SHRINKAGE_K = float(category_calib["n_fraud_val"].median())
category_calib["shrink_weight"] = category_calib["n_fraud_val"] / (category_calib["n_fraud_val"] + SHRINKAGE_K)
category_calib["threshold"] = (
    category_calib["shrink_weight"] * category_calib["raw_threshold"]
    + (1 - category_calib["shrink_weight"]) * best_threshold
)
category_thresholds = category_calib["threshold"].to_dict()

print(f"Shrinkage K (median fraud count in val): {SHRINKAGE_K:.1f}")
category_calib.sort_values("threshold")

Shrinkage K (median fraud count in val): 24.5


,n_legit_val,n_fraud_val,raw_threshold,shrink_weight,threshold
category,,,,,
grocery_pos,14732,162,0.000496,0.868633,0.005976
gas_transport,15977,52,0.001101,0.679739,0.014269
entertainment,11352,27,0.004790,0.524272,0.022594
home,15041,23,0.007903,0.484211,0.025602
shopping_pos,14031,78,0.026559,0.760976,0.030301
health_fitness,10100,12,0.031629,0.328767,0.038735
food_dining,11016,16,0.050025,0.395062,0.045301
grocery_net,5620,13,0.068790,0.346667,0.051428
kids_pets,13633,13,0.070756,0.346667,0.052110


In [5]:
test_eval = test_aug.copy()
test_eval["proba"] = test_proba
test_eval["threshold"] = test_eval["category"].map(category_thresholds).fillna(best_threshold)
test_eval["pred"] = (test_eval["proba"] >= test_eval["threshold"]).astype(int)

cm = confusion_matrix(y_test, test_eval["pred"])
print("Confusion matrix @ category-aware thresholds:")
print(cm)
print()
print(classification_report(y_test, test_eval["pred"], target_names=["legit", "fraud"], digits=3))

fn_mask_final = (test_eval.pred == 0) & (y_arr == 1)
fp_mask_final = (test_eval.pred == 1) & (y_arr == 0)
cost_category_aware = amounts[fn_mask_final].sum() + REVIEW_COST * fp_mask_final.sum()
overall_specificity_final = 1 - fp_mask_final.sum() / (y_arr == 0).sum()
overall_recall_final = 1 - fn_mask_final.sum() / y_arr.sum()

print(f"\nOverall specificity (category-aware): {overall_specificity_final:.4f}")
print(f"Overall recall (category-aware):      {overall_recall_final:.4f}")
print(f"Expected cost @ category-aware thresholds: ${cost_category_aware:,.0f}")
print(f"Estimated savings vs. no model: ${no_model_cost - cost_category_aware:,.0f} "
      f"({(1 - cost_category_aware / no_model_cost):.1%} reduction)")

by_cat = test_eval[test_eval.is_fraud == 1].groupby("category").agg(n=("pred", "size"), caught=("pred", "sum"))
by_cat["recall"] = by_cat.caught / by_cat.n
by_cat = by_cat[by_cat.n >= 5].sort_values("recall", ascending=False)
by_cat

Confusion matrix @ category-aware thresholds:
[[154206   2127]
 [    47    906]]

              precision    recall  f1-score   support

       legit      1.000     0.986     0.993    156333
       fraud      0.299     0.951     0.455       953

    accuracy                          0.986    157286
   macro avg      0.649     0.969     0.724    157286
weighted avg      0.995     0.986     0.990    157286


Overall specificity (category-aware): 0.9864
Overall recall (category-aware):      0.9507
Expected cost @ category-aware thresholds: $15,273
Estimated savings vs. no model: $496,010 (97.0% reduction)


,n,caught,recall
category,,,
shopping_net,215,215,1.000000
grocery_pos,230,229,0.995652
misc_net,137,136,0.992701
shopping_pos,96,95,0.989583
gas_transport,68,67,0.985294
home,30,28,0.933333
entertainment,28,26,0.928571
health_fitness,15,13,0.866667
grocery_net,17,14,0.823529


## Comparison: deployed (pre-stacking) vs. deployment-safe stacked

Loads the currently-live `reports/evaluation_summary.json` (pre-stacking, pure XGBoost on
21 features) to compare against this notebook's honest, val-calibrated stacked numbers --
not the 04e research number, which used val as training data.

In [6]:
with open("../reports/evaluation_summary.json") as f:
    prev_summary = json.load(f)

comparison = pd.DataFrame([
    {"model": "deployed (pre-stacking, 21 features)",
     "test_pr_auc": prev_summary["test_pr_auc"], "test_roc_auc": prev_summary["test_roc_auc"],
     "achieved_specificity": prev_summary["achieved_specificity"],
     "achieved_recall": prev_summary["achieved_recall"],
     "savings": prev_summary["estimated_savings_vs_no_model"]},
    {"model": "deployment-safe stacked (22 features, +nn_score)",
     "test_pr_auc": pr_auc, "test_roc_auc": roc_auc,
     "achieved_specificity": overall_specificity_final, "achieved_recall": overall_recall_final,
     "savings": no_model_cost - cost_category_aware},
]).set_index("model")
comparison

,test_pr_auc,test_roc_auc,achieved_specificity,achieved_recall,savings
model,,,,,
"deployed (pre-stacking, 21 features)",0.721958,0.990348,0.986516,0.900315,476872.93
"deployment-safe stacked (22 features, +nn_score)",0.871568,0.996490,0.986394,0.950682,496009.93


## Save deployment artifacts

Overwrites `models/fraud_xgboost.json`, `models/feature_columns.json` (now 22 features,
`nn_score` last), and `models/threshold.json` -- the exact three files `src/api/model.py`
loads at startup. `data/processed/feature_columns.json` (21-feature research config) is
left untouched.

In [7]:
stacked_model.save_model("../models/fraud_xgboost.json")

with open("../models/feature_columns.json", "w") as f:
    json.dump({
        "features": AUG_FEATURES, "label": LABEL,
        "mlflow_run_id": best_run_id, "mlflow_model_version": registered.version,
        "stacked": True,
        "nn_service_required": True,
    }, f, indent=2)

with open("../models/threshold.json", "w") as f:
    json.dump({
        "decision_threshold": float(best_threshold),
        "category_thresholds": {k: float(v) for k, v in category_thresholds.items()},
    }, f, indent=2)

deploy_summary = {
    "test_roc_auc": float(roc_auc),
    "test_pr_auc": float(pr_auc),
    "decision_threshold": float(best_threshold),
    "target_specificity": TARGET_SPECIFICITY,
    "achieved_specificity": float(overall_specificity_final),
    "achieved_recall": float(overall_recall_final),
    "cost_minimizing_threshold_reference": float(cost_min_threshold),
    "category_thresholds": {k: float(v) for k, v in category_thresholds.items()},
    "review_cost_per_fp": REVIEW_COST,
    "estimated_savings_vs_no_model": float(no_model_cost - cost_category_aware),
    "recall_by_category": by_cat["recall"].round(4).to_dict(),
    "n_test_transactions": int(len(test_aug)),
    "test_fraud_rate": float(y_test.mean()),
    "model_type": "xgboost_stacked_with_nn_score",
    "train_rows": int(len(X_train)),
    "mlflow_run_id": best_run_id,
    "mlflow_model_version": registered.version,
}
with open("../reports/evaluation_summary.json", "w") as f:
    json.dump(deploy_summary, f, indent=2)

print("Saved models/fraud_xgboost.json, models/feature_columns.json, models/threshold.json")
print("Saved reports/evaluation_summary.json (now reflects the deployed stacked model)")
print(json.dumps(deploy_summary, indent=2))

Saved models/fraud_xgboost.json, models/feature_columns.json, models/threshold.json
Saved reports/evaluation_summary.json (now reflects the deployed stacked model)
{
  "test_roc_auc": 0.9964899937912688,
  "test_pr_auc": 0.8715680380524612,
  "decision_threshold": 0.04221602529287338,
  "target_specificity": 0.98,
  "achieved_specificity": 0.9863944272802287,
  "achieved_recall": 0.950682056663169,
  "cost_minimizing_threshold_reference": 0.21000000000000002,
  "category_thresholds": {
    "entertainment": 0.022594390183831883,
    "food_dining": 0.045301091063905646,
    "gas_transport": 0.014268818652269496,
    "grocery_net": 0.05142828613519669,
    "grocery_pos": 0.005976213895074084,
    "health_fitness": 0.03873549411966376,
    "home": 0.025601512977951452,
    "kids_pets": 0.05210976292689641,
    "misc_net": 0.1193456215567367,
    "misc_pos": 0.07952397299566488,
    "personal_care": 0.0569624584352616,
    "shopping_net": 0.07813458310444946,
    "shopping_pos": 0.030301364